In [1]:
import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, Flatten, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

2026-02-08 22:41:09.611039: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-08 22:41:09.647695: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-08 22:41:10.446091: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
DATA_PATH = "/ravdess"

In [3]:
emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

In [ ]:
def extract_features(file_path):
    
    audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast', duration=2.5, offset=0.5)
    
    mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    
    mfccs_processed = np.mean(mfccs.T, axis=0)
    
    return mfccs_processed

In [ ]:
features = []
labels = []

print("Starting feature extraction...")

for actor_dir in os.listdir(DATA_PATH):
    if 'Actor_' in actor_dir:
        actor_path = os.path.join(DATA_PATH, actor_dir)
        for file_name in os.listdir(actor_path):
            if file_name.endswith(".wav"):
               
                part = file_name.split('-')
                emotion_code = part[2]
                emotion_label = emotion_map[emotion_code]
                
                file_path = os.path.join(actor_path, file_name)
                try:
                    data = extract_features(file_path)
                    features.append(data)
                    labels.append(emotion_label)
                except Exception as e:
                    print(f"Error handling {file_path}: {e}")

print(f"Extracted features for {len(features)} files.")

Starting feature extraction...
Extracted features for 1440 files.


In [ ]:
X = np.array(features)
y = np.array(labels)


lb = LabelEncoder()
y = to_categorical(lb.fit_transform(y))


print("Label Mapping:", dict(zip(lb.classes_, range(len(lb.classes_)))))
np.save('emotion_classes.npy', lb.classes_)


X = np.expand_dims(X, axis=2)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Label Mapping: {np.str_('angry'): 0, np.str_('calm'): 1, np.str_('disgust'): 2, np.str_('fearful'): 3, np.str_('happy'): 4, np.str_('neutral'): 5, np.str_('sad'): 6, np.str_('surprised'): 7}


In [ ]:
model = Sequential([
    
    Conv1D(64, kernel_size=5, activation='relu', input_shape=(40, 1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
    
    Conv1D(128, kernel_size=5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
   
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(8, activation='softmax') 
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

/home/jawahar-linux/workspace/.venv13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1770570784.739431  113044 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2153 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 36, 64)         │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 36, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 14, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 7, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 896)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       229,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 273,928 (1.04 MB)

 Trainable params: 273,544 (1.04 MB)

 Non-trainable params: 384 (1.50 KB)

In [8]:
history = model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=32, 
    validation_data=(X_test, y_test)
)

Epoch 1/50


2026-02-08 22:43:16.370177: I external/local_xla/xla/service/service.cc:163] XLA service 0x7050c0002570 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-08 22:43:16.370193: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-02-08 22:43:16.400210: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-08 22:43:16.625110: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-02-08 22:43:16.770539: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-08 22:43:17.

28/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1946 - loss: 3.0541 

I0000 00:00:1770570800.467932  114084 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.1936 - loss: 2.5949 - val_accuracy: 0.1771 - val_loss: 2.0277
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2804 - loss: 1.9479 - val_accuracy: 0.2674 - val_loss: 1.8627
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3299 - loss: 1.7905 - val_accuracy: 0.3264 - val_loss: 1.7736
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3507 - loss: 1.7383 - val_accuracy: 0.3576 - val_loss: 1.7576
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3889 - loss: 1.6675 - val_accuracy: 0.3958 - val_loss: 1.6106
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4097 - loss: 1.5848 - val_accuracy: 0.3993 - val_loss: 1.5571
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4280 - loss: 1.5363 - val_accuracy: 0.4757 - val_loss: 1.4973
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4635 - loss: 1.4483 - val_accuracy: 0.4861 - val_loss: 1.4264
Ep

In [12]:
model.save('my_audio_emotion_model.h5')
print("Audio Model Saved Successfully!")

Audio Model Saved Successfully!
